## Import

In [1]:
import kagglehub
from pathlib import Path

data_dir = Path("./Data")

if not data_dir.exists() or not any(data_dir.iterdir()):
    path = Path(kagglehub.dataset_download("kmader/finding-lungs-in-ct-data"))
    print("Dataset downloaded to:", path)
else:
    path = data_dir
    print("Dataset already exists at:", path.resolve())

Dataset downloaded to: /Users/jacob_kitz/.cache/kagglehub/datasets/kmader/finding-lungs-in-ct-data/versions/2


In [2]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from sklearn.model_selection import train_test_split

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

PyTorch: 2.13.0
Torchvision: 0.28.0


## Helpful functions

In [10]:
def show_slice(image, segmentation=None):

    # Handle PyTorch tensors
    if isinstance(image, torch.Tensor):
        im = image.squeeze().detach().cpu().numpy()
        seg = segmentation.squeeze().detach().cpu().numpy() if segmentation is not None else None

    # Handle SimpleITK images
    else:
        im = sitk.GetArrayFromImage(image)
        seg = sitk.GetArrayFromImage(segmentation) if segmentation is not None else None


    plt.figure(figsize=(7, 7))
    plt.imshow(im, cmap="gray")

    if seg is not None:
        plt.imshow(
            np.ma.masked_where(seg == 0, seg),
            cmap="autumn",
            alpha=0.5
        )

    plt.axis("off")
    plt.colorbar()
    plt.show()

## Dataset

In [4]:
image_dir = Path(path) / "3d_images"

image_files = sorted(image_dir.glob("IMG_*.nii.gz"))
mask_files = sorted(image_dir.glob("MASK_*.nii.gz"))

In [5]:
images = {
    f.name.replace("IMG_", "").replace(".nii.gz", ""): f
    for f in image_files
}

masks = {
    f.name.replace("MASK_", "").replace(".nii.gz", ""): f
    for f in mask_files
}

assert images.keys() == masks.keys(), "ids don't match"

volume_pairs = [
    (images[volume_id], masks[volume_id])
    for volume_id in sorted(images)
]

print(f"Successfully paired {len(volume_pairs)} volumes.")

Successfully paired 4 volumes.


In [16]:
class LungDataset(Dataset):

    def __init__(self, volume_pairs):
        self.images = []
        self.masks = []
        self.slice_indices = []

        for image_file, mask_file in volume_pairs:

            image = sitk.ReadImage(str(image_file))
            mask = sitk.ReadImage(str(mask_file))

            # preprocessing
            image = sitk.Normalize(image)

            image = sitk.GetArrayFromImage(image).astype(np.float32)
            mask = sitk.GetArrayFromImage(mask).astype(np.int32)
            mask = (mask > 0).astype(np.uint8)

            self.images.append(image)
            self.masks.append(mask)

            volume_idx = len(self.images) - 1

            for z in range(image.shape[2]):
                self.slice_indices.append((volume_idx, z))

    def __len__(self):
        return len(self.slice_indices)

    def __getitem__(self, idx):

        volume_idx, slice_idx = self.slice_indices[idx]

        image = self.images[volume_idx][:, :, slice_idx]
        mask = self.masks[volume_idx][:, :, slice_idx]

        image = torch.from_numpy(image.copy()).float().unsqueeze(0)
        mask = torch.from_numpy(mask.copy()).long()

        return image, mask

In [ ]:
dataset = LungDataset(volume_pairs)

print("Total slices:", len(dataset))

image, mask = dataset[500]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)

show_slice(image, mask)

## Split

In [8]:
train_pairs, val_pairs = train_test_split(
    volume_pairs,
    test_size=0.25,
    random_state=42
)

train_dataset = LungDataset(train_pairs)
val_dataset = LungDataset(val_pairs)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Training volumes:", len(train_pairs))
print("Validation volumes:", len(val_pairs))
print("Training slices:", len(train_dataset))
print("Validation slices:", len(val_dataset))

Training volumes: 3
Validation volumes: 1
Training slices: 743
Validation slices: 465


## Check dataset preprocessing

In [9]:
for i, (image_file, mask_file) in enumerate(volume_pairs):

    image = sitk.GetArrayFromImage(
        sitk.ReadImage(str(image_file))
    )

    mask = sitk.GetArrayFromImage(
        sitk.ReadImage(str(mask_file))
    )

    print(f"\nVolume {i}: {image_file.name}")
    print(f"Image range: {image.min()} to {image.max()}")
    print(f"Image percentiles: {np.percentile(image, [1, 25, 50, 75, 99])}")
    print(f"Mask labels: {np.unique(mask)}")


Volume 0: IMG_0002.nii.gz
Image range: -933 to 2675
Image percentiles: [-913. -806. -716.  -16.  446.]
Mask labels: [  0 255]

Volume 1: IMG_0031.nii.gz
Image range: -2619 to 1508
Image percentiles: [-2.576e+03 -8.590e+02 -5.800e+02 -1.000e+00  4.430e+02]
Mask labels: [  0 255]

Volume 2: IMG_0059.nii.gz
Image range: -953 to 2658
Image percentiles: [-911. -769. -389.   -3.  354.]
Mask labels: [  0 255]

Volume 3: IMG_0078.nii.gz
Image range: -1806 to 2655
Image percentiles: [-1770.  -868.  -675.   -23.   416.]
Mask labels: [  0 255]
